# Advanced Problems with Solutions: Boolean Operators in Python

Topics covered: truthiness, operand-returning behavior, short-circuiting, `and`, `or`, `not`, default values, safe access patterns, and common traps.

## Problem 1 — Predict the exact results

For each expression, predict the exact value and type returned.

```python
expressions = [
    [] or "fallback",
    [0] or "fallback",
    "" and "next",
    "hello" and [],
    None or 0 or [] or "done",
    "start" and 0 and "end",
    not "",
    not [False],
]
```

### Solution 1

`and` and `or` do not necessarily return booleans.

- `X or Y` returns `X` if `X` is truthy, otherwise returns `Y`.
- `X and Y` returns `X` if `X` is falsy, otherwise returns `Y`.
- `not X` always returns a real boolean: `True` or `False`.

In [1]:
expressions = [
    [] or "fallback",
    [0] or "fallback",
    "" and "next",
    "hello" and [],
    None or 0 or [] or "done",
    "start" and 0 and "end",
    not "",
    not [False],
]

for value in expressions:
    print(repr(value), type(value).__name__)

'fallback' str
[0] list
'' str
[] list
'done' str
0 int
True bool
False bool


Expected output:

```text
'fallback' str
[0] list
'' str
[] list
'done' str
0 int
True bool
False bool
```

## Problem 2 — Short-circuiting and dangerous expressions

Which of these expressions raise an exception?

```python
1 or 1 / 0
0 or 1 / 0
0 and 1 / 0
1 and 1 / 0
[] and [][0]
[1] and [][0]
```

### Solution 2

Only expressions that need to evaluate the dangerous right-hand side raise an exception.

```python
1 or 1 / 0      # returns 1
0 or 1 / 0      # raises ZeroDivisionError
0 and 1 / 0     # returns 0
1 and 1 / 0     # raises ZeroDivisionError
[] and [][0]    # returns []
[1] and [][0]   # raises IndexError
```

In [2]:
tests = [
    "1 or 1 / 0",
    "0 or 1 / 0",
    "0 and 1 / 0",
    "1 and 1 / 0",
    "[] and [][0]",
    "[1] and [][0]",
]

for expr in tests:
    try:
        print(expr, "=>", eval(expr))
    except Exception as e:
        print(expr, "=>", type(e).__name__)

1 or 1 / 0 => 1
0 or 1 / 0 => ZeroDivisionError
0 and 1 / 0 => 0
1 and 1 / 0 => ZeroDivisionError
[] and [][0] => []
[1] and [][0] => IndexError


## Problem 3 — Implement lazy versions of `and` and `or`

The following functions are not equivalent to real `and` and `or` because function arguments are evaluated before the function call:

```python
def eager_or(x, y):
    return x if x else y

def eager_and(x, y):
    return y if x else x
```

Write `lazy_or` and `lazy_and` that accept two zero-argument functions and preserve short-circuiting.

### Solution 3

We delay evaluation by passing callables instead of already-computed values.

In [3]:
def lazy_or(x_func, y_func):
    x = x_func()
    if x:
        return x
    return y_func()


def lazy_and(x_func, y_func):
    x = x_func()
    if not x:
        return x
    return y_func()


print(lazy_or(lambda: 1, lambda: 1 / 0))      # 1
print(lazy_and(lambda: 0, lambda: 1 / 0))     # 0
print(lazy_or(lambda: "", lambda: "backup")) # backup
print(lazy_and(lambda: "abc", lambda: "ok")) # ok

1
0
backup
ok


## Problem 4 — Safe nested lookup

Given possibly incomplete dictionaries, safely get the user email.

Return `'missing'` if any step is missing or falsy.

```python
records = [
    {'user': {'email': 'a@example.com'}},
    {'user': {'email': ''}},
    {'user': {}},
    {'user': None},
    {},
]
```

### Solution 4

Use `and` to safely continue only while the previous value is truthy, then use `or` for the default.

In [4]:
records = [
    {'user': {'email': 'a@example.com'}},
    {'user': {'email': ''}},
    {'user': {}},
    {'user': None},
    {},
]

for record in records:
    email = (record.get('user') and record.get('user').get('email')) or 'missing'
    print(email)

a@example.com
missing
missing
missing
missing


Expected output:

```text
a@example.com
missing
missing
missing
missing
```

Note: this treats empty strings as missing. That may or may not be what you want.

## Problem 5 — Fix the default-value bug

This function is intended to return a user-provided limit, or use `10` only when the user did not provide one.

```python
def get_limit(user_limit):
    return user_limit or 10
```

Why is this wrong? Fix it.

### Solution 5

`0` is falsy, but `0` may be a valid value. The expression `user_limit or 10` incorrectly replaces `0` with `10`.

Use an explicit `None` check when `None` means “not provided”.

In [5]:
def get_limit(user_limit):
    return 10 if user_limit is None else user_limit


assert get_limit(None) == 10
assert get_limit(0) == 0
assert get_limit(5) == 5

print("All tests passed.")

All tests passed.


## Problem 6 — Trace evaluation order

Without running the code first, predict the output.

```python
def mark(name, value):
    print(f'evaluating {name}')
    return value

result = mark('A', '') or mark('B', []) or mark('C', 'final')
print(result)
```

### Solution 6

`or` keeps evaluating until it finds a truthy value or reaches the final operand.

In [6]:
def mark(name, value):
    print(f'evaluating {name}')
    return value

result = mark('A', '') or mark('B', []) or mark('C', 'final')
print(result)

evaluating A
evaluating B
evaluating C
final


Expected output:

```text
evaluating A
evaluating B
evaluating C
final
```

## Problem 7 — Rewrite without Boolean operators

Rewrite this expression using only `if`, `else`, and temporary variables:

```python
result = a and b or c
```

Then explain why this expression is not always equivalent to a ternary expression.

### Solution 7

`and` has higher precedence than `or`, so this means:

```python
result = (a and b) or c
```

Expanded version:

In [7]:
def expanded(a, b, c):
    if a:
        temp = b
    else:
        temp = a

    if temp:
        return temp
    else:
        return c


print(expanded(True, "B", "C"))
print(expanded(True, "", "C"))
print(expanded(False, "B", "C"))

B
C
C


Important trap:

```python
a and b or c
```

is sometimes used as an old-style replacement for:

```python
b if a else c
```

But they differ when `b` is falsy.

In [8]:
a = True
b = ""
c = "fallback"

print(a and b or c)      # fallback
print(b if a else c)     # empty string

fallback



## Problem 8 — Custom truthiness

Create a class `Cart` whose instances are truthy only when they contain at least one item.

Then use Boolean operators to print either the cart itself or `'empty cart'`.

### Solution 8

Python uses `__bool__` to determine custom truthiness. If `__bool__` is not defined, Python may use `__len__`.

In [9]:
class Cart:
    def __init__(self, items):
        self.items = list(items)

    def __bool__(self):
        return len(self.items) > 0

    def __repr__(self):
        return f"Cart({self.items!r})"


c1 = Cart([])
c2 = Cart(['book', 'pen'])

print(c1 or 'empty cart')
print(c2 or 'empty cart')

empty cart
Cart(['book', 'pen'])


## Problem 9 — Build a safe first-character function

Write a function `first_char(value, default='n/a')` that:

- returns the first character if `value` is a non-empty string,
- returns `default` if `value` is `None` or an empty string,
- does not raise an error for `None`.

### Solution 9

This is a compact use of `and` and `or`.

In [10]:
def first_char(value, default='n/a'):
    return (value and value[0]) or default


assert first_char(None) == 'n/a'
assert first_char('') == 'n/a'
assert first_char('abc') == 'a'

print("All tests passed.")

All tests passed.


Caution: this works well here because if `value` is a non-empty string, `value[0]` is also a non-empty string. If the middle result could be falsy, this pattern may accidentally return the default.

## Problem 10 — Advanced challenge: explain each returned object

Predict the exact output and explain why each line returns that object.

```python
values = [None, '', 'python', [], [1], 0, 42]

for x in values:
    print(repr(x), '=>', repr((x and 'truthy') or 'falsy'))
```

### Solution 10

The expression is:

```python
(x and 'truthy') or 'falsy'
```

If `x` is falsy, `x and 'truthy'` returns `x`, then `x or 'falsy'` returns `'falsy'`.

If `x` is truthy, `x and 'truthy'` returns `'truthy'`, then `'truthy' or 'falsy'` returns `'truthy'`.

In [11]:
values = [None, '', 'python', [], [1], 0, 42]

for x in values:
    print(repr(x), '=>', repr((x and 'truthy') or 'falsy'))

None => 'falsy'
'' => 'falsy'
'python' => 'truthy'
[] => 'falsy'
[1] => 'truthy'
0 => 'falsy'
42 => 'truthy'


Expected output:

```text
None => 'falsy'
'' => 'falsy'
'python' => 'truthy'
[] => 'falsy'
[1] => 'truthy'
0 => 'falsy'
42 => 'truthy'
```

## Summary

Best practices:

- Use `x or default` only when all falsy values should be replaced.
- Use `x is None` when only `None` means missing.
- Remember that `and` and `or` return operands, not necessarily booleans.
- Use short-circuiting for safe access, but avoid making expressions too clever.
- Use the ternary expression `a if condition else b` when choosing between two values explicitly.